# Stage 1: paired dataset generation
No GPU needed. Nothing to upload: this notebook writes the scripts itself.

Run the cells **in order, top to bottom**.

### Cell 1: mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/fyp_dna'
CODE    = '/content/drive/MyDrive/fyp_dna/code'
REPO    = '/content/Deep-DNA-based-storage'
DATA    = f'{PROJECT}/data/stage1_pilot'
TINY    = '/content/stage1_tiny'
os.makedirs(CODE, exist_ok=True)
os.makedirs(f'{PROJECT}/logs', exist_ok=True)
print('ready')

### Cell 2: write the dataset generator script to Drive

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage1_generate.py
"""
Stage 1: paired dataset generator for the marker vs. marker-free DNAFormer study.

Makes three paired "arms" from the SAME random payloads and the SAME noise seeds:
  marker_free      128 payload bases, no markers
  marker_periodic  (24 payload + 8 marker) x 4   = 96 payload + 32 marker
  marker_endblock  96 payload, then the 4 markers together at the end (control)

Noisy reads come from DNAFormer's own simulator
(Deep-DNA-based-storage/DataGenerator/cluster_generator.py). We only give it our
own error rates; we do not edit the repository.

Usage (Colab or local):
  python stage1_generate.py --repo /content/Deep-DNA-based-storage --out /content/drive/MyDrive/fyp_dna/data/stage1_pilot
  python stage1_generate.py --repo ... --out ... --tiny      # quick smoke test
"""
import argparse
import csv
import hashlib
import json
import os
import random
import subprocess
import sys
import time

import numpy as np

# ---------------------------------------------------------------------------
# Fixed experiment settings (saved to config.json)
# ---------------------------------------------------------------------------
BASES = "ACGT"
STRAND_LEN = 128
MARKER = "ACGTCAGT"
MARKER_INTERVAL = 24          # payload bases between markers
N_MARKERS = 4
N_READS = 10
DELETION_LEVELS = [0.01, 0.03, 0.05, 0.08]
INSERTION_RATE = 0.01
SUBSTITUTION_RATE = 0.01
LONG_DELETION_RATE = 0.0      # all deletions are single-base; change later if wanted

ARMS = ["marker_free", "marker_periodic", "marker_endblock"]

# Seed ranges never overlap between splits, so the same payload cannot appear in two splits.
SPLIT_SEED_BASE = {"train": 10_000_000, "val": 20_000_000, "test": 30_000_000}
LEVEL_SEED_STRIDE = 1_000_000  # each deletion level gets its own block of seeds
NOISE_SEED_OFFSET = 1_000_000_000
SHUFFLE_SEED = 12345

PILOT_SIZES = {"train": 2500, "val": 250, "test": 1000}   # clusters PER deletion level
TINY_SIZES = {"train": 20, "val": 5, "test": 10}


def level_tag(d):
    return "del%02d" % round(d * 100)


# ---------------------------------------------------------------------------
# Strand construction
# ---------------------------------------------------------------------------
def make_payload128(payload_seed):
    rng = np.random.default_rng(payload_seed)
    return "".join(BASES[i] for i in rng.integers(0, 4, STRAND_LEN))


def build_strand(arm, payload128):
    """Return (clean_strand, payload, marker_positions)."""
    if arm == "marker_free":
        return payload128, payload128, []
    payload96 = payload128[: MARKER_INTERVAL * N_MARKERS]  # same seed -> first 96 bases shared
    if arm == "marker_periodic":
        parts, positions = [], []
        for k in range(N_MARKERS):
            parts.append(payload96[k * MARKER_INTERVAL:(k + 1) * MARKER_INTERVAL])
            start = sum(len(p) for p in parts)
            parts.append(MARKER)
            positions.append([start, start + len(MARKER)])
        return "".join(parts), payload96, positions
    if arm == "marker_endblock":
        block = MARKER * N_MARKERS
        start = len(payload96)
        positions = [[start + k * len(MARKER), start + (k + 1) * len(MARKER)] for k in range(N_MARKERS)]
        return payload96 + block, payload96, positions
    raise ValueError(arm)


def remove_markers(arm, strand):
    """Payload-only view of a 128-base strand (used by later stages for scoring)."""
    if arm == "marker_free":
        return strand
    if arm == "marker_periodic":
        step = MARKER_INTERVAL + len(MARKER)
        return "".join(strand[k * step:k * step + MARKER_INTERVAL] for k in range(N_MARKERS))
    if arm == "marker_endblock":
        return strand[: MARKER_INTERVAL * N_MARKERS]
    raise ValueError(arm)


def payload_to_bytes(payload):
    """2 bits per base (A=00 C=01 G=10 T=11). 128 bases -> 32 bytes, 96 bases -> 24 bytes."""
    bits = "".join(format(BASES.index(b), "02b") for b in payload)
    return bytes(int(bits[i:i + 8], 2) for i in range(0, len(bits), 8))


# ---------------------------------------------------------------------------
# DNAFormer simulator wrapper
# ---------------------------------------------------------------------------
def load_simulator(repo):
    gen_dir = os.path.join(repo, "DataGenerator")
    if not os.path.isfile(os.path.join(gen_dir, "cluster_generator.py")):
        sys.exit("ERROR: cluster_generator.py not found in %s. Check --repo." % gen_dir)
    sys.path.insert(0, gen_dir)
    from cluster_generator import ClusterGenerator  # noqa: E402
    return ClusterGenerator


def error_rates(deletion_rate):
    """Rates in the simulator's own format.

    The simulator gives each base one error chance equal to the SUM of the general
    rates, then picks the error type with the per-base weights. Using the same numbers
    for both makes the per-base probabilities exactly sub/ins/del.
    """
    general = {"s": SUBSTITUTION_RATE, "i": INSERTION_RATE, "d": deletion_rate, "ld": LONG_DELETION_RATE}
    per_base = {b: {"s": SUBSTITUTION_RATE, "i": 0.25, "pi": INSERTION_RATE,
                    "d": deletion_rate, "ld": LONG_DELETION_RATE} for b in BASES}
    # 'i' only chooses WHICH base is inserted (uniform here); 'pi' is the insertion rate.
    return general, per_base


def simulate_reads(ClusterGenerator, clean, deletion_rate, noise_seed):
    random.seed(noise_seed)          # the simulator uses Python's global random module
    np.random.seed(noise_seed % (2 ** 32))
    general, per_base = error_rates(deletion_rate)
    gen = ClusterGenerator(general, per_base, clean, min_copies=N_READS, max_copies=N_READS)
    gen.cluster_size = N_READS       # the repo's randint(min, max+1) could give 11; force exactly 10
    gen.generate_cluster(delta=0)    # delta=0: no random jitter of the rates
    assert len(gen.copies) == N_READS
    return list(gen.copies)


# ---------------------------------------------------------------------------
# Cluster plan
# ---------------------------------------------------------------------------
def plan_clusters(sizes):
    """List of dicts: split, subset, deletion_rate, payload_seed, noise_seed, pair_id."""
    plan = []
    for split in ["train", "val", "test"]:
        for li, d in enumerate(DELETION_LEVELS):
            for i in range(sizes[split]):
                seed = SPLIT_SEED_BASE[split] + li * LEVEL_SEED_STRIDE + i
                subset = split if split != "test" else "test_" + level_tag(d)
                plan.append({
                    "pair_id": "%s_%s_%06d" % (split, level_tag(d), i),
                    "split": split,
                    "subset": subset,
                    "deletion_rate": d,
                    "payload_seed": seed,
                    "noise_seed": seed + NOISE_SEED_OFFSET,
                })
    # Mix the deletion levels inside train and val (fixed shuffle seed).
    rng = random.Random(SHUFFLE_SEED)
    for split in ["train", "val"]:
        idx = [k for k, c in enumerate(plan) if c["split"] == split]
        items = [plan[k] for k in idx]
        rng.shuffle(items)
        for k, c in zip(idx, items):
            plan[k] = c
    return plan


def make_cluster(ClusterGenerator, arm, c):
    payload128 = make_payload128(c["payload_seed"])
    clean, payload, marker_pos = build_strand(arm, payload128)
    reads = simulate_reads(ClusterGenerator, clean, c["deletion_rate"], c["noise_seed"])
    n_marker = len(clean) - len(payload)
    return {
        "cluster_id": "%s__%s" % (arm, c["pair_id"]),
        "pair_id": c["pair_id"],
        "arm": arm,
        "split": c["split"],
        "subset": c["subset"],
        "seed": c["payload_seed"],
        "payload_seed": c["payload_seed"],
        "noise_seed": c["noise_seed"],
        "marker_present": arm != "marker_free",
        "marker_layout": {"marker_free": "none", "marker_periodic": "periodic",
                          "marker_endblock": "end_block"}[arm],
        "marker_sequence": MARKER if arm != "marker_free" else "",
        "marker_interval": MARKER_INTERVAL if arm == "marker_periodic" else None,
        "marker_positions": marker_pos,
        "marker_overhead": n_marker / len(clean),
        "payload_length": len(payload),
        "strand_length": len(clean),
        "clean_strand": clean,
        "payload_strand": payload,
        "payload_bytes_hex": payload_to_bytes(payload).hex(),
        "noisy_reads": reads,
        "read_lengths": [len(r) for r in reads],
        "deletion_rate": c["deletion_rate"],
        "insertion_rate": INSERTION_RATE,
        "substitution_rate": SUBSTITUTION_RATE,
        "long_deletion_rate": LONG_DELETION_RATE,
    }


# ---------------------------------------------------------------------------
# Writing
# ---------------------------------------------------------------------------
def write_subset(folder, clusters):
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, "clean.fasta"), "w", newline="\n") as f:
        for c in clusters:
            f.write(">%s\n%s\n" % (c["cluster_id"], c["clean_strand"]))
    with open(os.path.join(folder, "noisy_reads.fasta"), "w", newline="\n") as f:
        for c in clusters:
            for j, r in enumerate(c["noisy_reads"]):
                f.write(">%s|read_%d\n%s\n" % (c["cluster_id"], j, r))
    with open(os.path.join(folder, "metadata.json"), "w", newline="\n") as f:
        json.dump(clusters, f, separators=(",", ":"))


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def repo_commit(repo):
    try:
        return subprocess.check_output(["git", "-C", repo, "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True, help="path to Deep-DNA-based-storage")
    ap.add_argument("--out", required=True, help="output folder")
    ap.add_argument("--tiny", action="store_true", help="very small smoke-test dataset")
    args = ap.parse_args()

    sizes = TINY_SIZES if args.tiny else PILOT_SIZES
    ClusterGenerator = load_simulator(args.repo)
    plan = plan_clusters(sizes)
    os.makedirs(args.out, exist_ok=True)

    t0 = time.time()
    manifest_rows = []
    for arm in ARMS:
        by_subset = {}
        for k, c in enumerate(plan):
            cl = make_cluster(ClusterGenerator, arm, c)
            by_subset.setdefault(c["subset"], []).append(cl)
            if (k + 1) % 2000 == 0:
                print("  %s: %d/%d clusters (%.0fs)" % (arm, k + 1, len(plan), time.time() - t0), flush=True)
        for subset, clusters in by_subset.items():
            folder = os.path.join(args.out, arm, subset)
            write_subset(folder, clusters)
            for cl in clusters:
                manifest_rows.append({
                    "cluster_id": cl["cluster_id"], "pair_id": cl["pair_id"], "arm": arm,
                    "split": cl["split"], "subset": subset,
                    "deletion_rate": cl["deletion_rate"], "insertion_rate": cl["insertion_rate"],
                    "substitution_rate": cl["substitution_rate"],
                    "payload_seed": cl["payload_seed"], "noise_seed": cl["noise_seed"],
                    "marker_present": cl["marker_present"], "marker_layout": cl["marker_layout"],
                    "marker_sequence": cl["marker_sequence"], "marker_interval": cl["marker_interval"],
                    "payload_length": cl["payload_length"], "strand_length": cl["strand_length"],
                    "marker_overhead": cl["marker_overhead"], "n_reads": len(cl["noisy_reads"]),
                    "mean_read_length": round(float(np.mean(cl["read_lengths"])), 2),
                    "folder": os.path.join(arm, subset).replace("\\", "/"),
                })
        print("%s done (%.0fs)" % (arm, time.time() - t0), flush=True)

    with open(os.path.join(args.out, "dataset_manifest.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(manifest_rows[0].keys()))
        w.writeheader()
        w.writerows(manifest_rows)

    config = {
        "stage": 1, "mode": "tiny" if args.tiny else "pilot",
        "clusters_per_level": sizes, "deletion_levels": DELETION_LEVELS,
        "insertion_rate": INSERTION_RATE, "substitution_rate": SUBSTITUTION_RATE,
        "long_deletion_rate": LONG_DELETION_RATE, "strand_length": STRAND_LEN,
        "marker": MARKER, "marker_interval": MARKER_INTERVAL, "n_markers": N_MARKERS,
        "n_reads": N_READS, "arms": ARMS, "split_seed_base": SPLIT_SEED_BASE,
        "level_seed_stride": LEVEL_SEED_STRIDE, "noise_seed_offset": NOISE_SEED_OFFSET,
        "shuffle_seed": SHUFFLE_SEED, "simulator": "DataGenerator/cluster_generator.py",
        "simulator_repo_commit": repo_commit(args.repo),
        "python": sys.version.split()[0], "numpy": np.__version__,
    }
    with open(os.path.join(args.out, "config.json"), "w", newline="\n") as f:
        json.dump(config, f, indent=2)

    checksums = {}
    for root, _, files in os.walk(args.out):
        for name in files:
            if name in ("checksums.json",):
                continue
            p = os.path.join(root, name)
            checksums[os.path.relpath(p, args.out).replace("\\", "/")] = sha256(p)
    with open(os.path.join(args.out, "checksums.json"), "w", newline="\n") as f:
        json.dump(dict(sorted(checksums.items())), f, indent=1)

    print("\nDONE: %d clusters per arm x %d arms in %.0fs -> %s"
          % (len(plan), len(ARMS), time.time() - t0, args.out))


if __name__ == "__main__":
    main()


### Cell 3: write the validation script to Drive

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage1_validate.py
"""
Stage 1 validation: checks the generated dataset and prints a summary to paste back.

Usage:
  python stage1_validate.py --repo /content/Deep-DNA-based-storage --data /content/drive/MyDrive/fyp_dna/data/stage1_pilot
"""
import argparse
import collections
import hashlib
import json
import os
import random
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import stage1_generate as G  # noqa: E402

FAILS = []


def check(ok, msg, quiet=False):
    if not ok or not quiet:
        print(("  [PASS] " if ok else "  [FAIL] ") + msg)
    if not ok:
        FAILS.append(msg)


def read_fasta(path):
    out, name = collections.OrderedDict(), None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                name = line[1:]
            elif line:
                out[name] = line
    return out


def align_counts(ref, read):
    """Edit-distance alignment; returns (substitutions, insertions, deletions)."""
    n, m = len(ref), len(read)
    D = np.zeros((n + 1, m + 1), dtype=np.int32)
    D[:, 0] = np.arange(n + 1)
    D[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + (ref[i - 1] != read[j - 1]))
    i, j, s, ins, dl = n, m, 0, 0, 0
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i - 1, j - 1] + (ref[i - 1] != read[j - 1]):
            s += ref[i - 1] != read[j - 1]
            i, j = i - 1, j - 1
        elif i > 0 and D[i, j] == D[i - 1, j] + 1:
            dl += 1
            i -= 1
        else:
            ins += 1
            j -= 1
    return s, ins, dl


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--align-sample", type=int, default=30, help="clusters per level for error-rate estimate")
    args = ap.parse_args()

    cfg = json.load(open(os.path.join(args.data, "config.json")))
    sizes = cfg["clusters_per_level"]
    levels = cfg["deletion_levels"]
    print("Dataset: %s  mode=%s  sizes per level=%s" % (args.data, cfg["mode"], sizes))
    print("Simulator commit: %s" % cfg["simulator_repo_commit"])

    # ---------------------------------------------------------------- load
    data = {}  # arm -> subset -> list of clusters
    print("\n1. Files and counts")
    subsets = ["train", "val"] + ["test_" + G.level_tag(d) for d in levels]
    for arm in G.ARMS:
        data[arm] = {}
        for sub in subsets:
            folder = os.path.join(args.data, arm, sub)
            files_ok = all(os.path.isfile(os.path.join(folder, f))
                           for f in ("clean.fasta", "noisy_reads.fasta", "metadata.json"))
            check(files_ok, "%s/%s has clean.fasta, noisy_reads.fasta, metadata.json" % (arm, sub), quiet=True)
            if not files_ok:
                continue
            meta = json.load(open(os.path.join(folder, "metadata.json")))
            data[arm][sub] = meta
            want = sizes["test"] if sub.startswith("test") else sizes[sub] * len(levels)
            check(len(meta) == want, "%s/%s: %d clusters (expected %d)" % (arm, sub, len(meta), want), quiet=True)
            clean_fa = read_fasta(os.path.join(folder, "clean.fasta"))
            reads_fa = read_fasta(os.path.join(folder, "noisy_reads.fasta"))
            fasta_ok = (len(clean_fa) == len(meta) and len(reads_fa) == len(meta) * G.N_READS and
                        all(clean_fa[c["cluster_id"]] == c["clean_strand"] for c in meta) and
                        all(reads_fa["%s|read_%d" % (c["cluster_id"], j)] == r
                            for c in meta for j, r in enumerate(c["noisy_reads"])))
            check(fasta_ok, "%s/%s: FASTA files match metadata.json" % (arm, sub), quiet=True)
        n_fail = len(FAILS)
        print("  [%s] %-16s %s" % ("PASS" if n_fail == 0 else "FAIL", arm,
              ", ".join("%s=%d" % (k, len(v)) for k, v in data[arm].items())))

    all_clusters = [c for arm in data for sub in data[arm] for c in data[arm][sub]]

    # ---------------------------------------------------------------- strands
    print("\n2. Strand structure and markers")
    check(all(len(c["clean_strand"]) == 128 for c in all_clusters), "every clean strand is 128 bases")
    check(all(set(c["clean_strand"]) <= set("ACGT") for c in all_clusters), "clean strands use only A/C/G/T")
    check(all(len(c["noisy_reads"]) == 10 for c in all_clusters), "every cluster has exactly 10 noisy reads")
    check(all(set(r) <= set("ACGT") for c in all_clusters for r in c["noisy_reads"]), "reads use only A/C/G/T")
    check(len({c["cluster_id"] for c in all_clusters}) == len(all_clusters), "cluster_ids are unique")

    per = [c for c in all_clusters if c["arm"] == "marker_periodic"]
    endb = [c for c in all_clusters if c["arm"] == "marker_endblock"]
    free = [c for c in all_clusters if c["arm"] == "marker_free"]
    check(all(all(c["clean_strand"][24 + 32 * k:32 + 32 * k] == G.MARKER for k in range(4)) for c in per),
          "periodic: marker ACGTCAGT at positions 24-31, 56-63, 88-95, 120-127")
    check(all(c["marker_positions"] == [[24, 32], [56, 64], [88, 96], [120, 128]] for c in per),
          "periodic: stored marker_positions are correct")
    check(all(c["clean_strand"][96:] == G.MARKER * 4 for c in endb), "end-block: 32 marker bases at positions 96-127")
    check(all(c["marker_positions"] == [] and not c["marker_present"] for c in free), "marker-free: no markers recorded")
    check(all(G.remove_markers(c["arm"], c["clean_strand"]) == c["payload_strand"] for c in all_clusters),
          "removing markers gives back the stored payload (all arms)")
    check(all(bytes.fromhex(c["payload_bytes_hex"]) == G.payload_to_bytes(c["payload_strand"]) for c in all_clusters),
          "payload_bytes_hex matches payload")

    # ---------------------------------------------------------------- pairing
    print("\n3. Pairing across arms")
    idx = {arm: {c["pair_id"]: c for sub in data[arm] for c in data[arm][sub]} for arm in data}
    pair_ids = set(idx["marker_free"])
    check(all(set(idx[a]) == pair_ids for a in G.ARMS), "all three arms contain the same pair_ids")
    paired_ok = True
    for p in pair_ids:
        f, q, e = idx["marker_free"][p], idx["marker_periodic"][p], idx["marker_endblock"][p]
        same = (f["payload_seed"] == q["payload_seed"] == e["payload_seed"] and
                f["noise_seed"] == q["noise_seed"] == e["noise_seed"] and
                f["deletion_rate"] == q["deletion_rate"] == e["deletion_rate"] and
                f["insertion_rate"] == q["insertion_rate"] == e["insertion_rate"] and
                f["substitution_rate"] == q["substitution_rate"] == e["substitution_rate"] and
                f["subset"] == q["subset"] == e["subset"] and
                q["payload_strand"] == e["payload_strand"] == f["payload_strand"][:96])
        paired_ok &= same
    check(paired_ok, "same seed, error rates, split and payload (marker arms = first 96 bases of marker-free payload)")

    # ---------------------------------------------------------------- leakage
    print("\n4. Leakage between splits")
    split_seeds = collections.defaultdict(set)
    split_payloads = collections.defaultdict(set)
    for c in free:
        split_seeds[c["split"]].add(c["payload_seed"])
        split_payloads[c["split"]].add(c["payload_strand"])
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        check(not (split_seeds[a] & split_seeds[b]), "no shared seeds between %s and %s" % (a, b))
        check(not (split_payloads[a] & split_payloads[b]), "no identical payloads between %s and %s" % (a, b))
    p96 = collections.defaultdict(set)
    for c in per:
        p96[c["split"]].add(c["payload_strand"])
    check(not (p96["train"] & (p96["val"] | p96["test"])), "no identical 96-base payloads leak from train")

    # ---------------------------------------------------------------- error levels
    print("\n5. Deletion levels per subset")
    for sub in ["train", "val"]:
        cnt = collections.Counter(c["deletion_rate"] for c in data["marker_free"].get(sub, []))
        check(sorted(cnt) == sorted(levels) and len(set(cnt.values())) == 1,
              "%s mixes all levels equally: %s" % (sub, dict(sorted(cnt.items()))))
    for d in levels:
        sub = "test_" + G.level_tag(d)
        rates = {c["deletion_rate"] for arm in G.ARMS for c in data[arm].get(sub, [])}
        check(rates == {d}, "%s contains only deletion rate %.2f" % (sub, d))
    check(all(c["insertion_rate"] == 0.01 and c["substitution_rate"] == 0.01 for c in all_clusters),
          "insertion and substitution fixed at 1% everywhere")

    # ---------------------------------------------------------------- reproducibility
    print("\n6. Reproducibility")
    ClusterGenerator = G.load_simulator(args.repo)
    rng = random.Random(0)
    sample = rng.sample(all_clusters, min(60, len(all_clusters)))
    plan_lookup = {c["pair_id"]: c for c in G.plan_clusters(sizes)}
    regen_ok = all(G.make_cluster(ClusterGenerator, c["arm"], plan_lookup[c["pair_id"]]) == c for c in sample)
    check(regen_ok, "regenerating %d random clusters from their seeds gives identical clusters" % len(sample))
    sums = json.load(open(os.path.join(args.data, "checksums.json")))
    bad = [p for p, h in sums.items()
           if hashlib.sha256(open(os.path.join(args.data, p), "rb").read()).hexdigest() != h]
    check(not bad, "all %d files match checksums.json" % len(sums))

    # ---------------------------------------------------------------- statistics
    print("\n7. Read statistics per deletion level (marker_free arm, test subsets)")
    print("   level | mean len | expected | measured sub/ins/del per base | reads DNAFormer would DROP (|len-128|>4) | reads >132")
    for d in levels:
        cl = data["marker_free"].get("test_" + G.level_tag(d), [])
        lens = np.array([len(r) for c in cl for r in c["noisy_reads"]])
        s = i = dl = 0
        for c in cl[:args.align_sample]:
            for r in c["noisy_reads"]:
                a, b, e = align_counts(c["clean_strand"], r)
                s, i, dl = s + a, i + b, dl + e
        nb = 128 * G.N_READS * min(len(cl), args.align_sample)
        print("   %4.0f%% | %8.2f | %8.2f | %.4f / %.4f / %.4f            | %5.1f%%                                   | %4.1f%%"
              % (d * 100, lens.mean(), 128 * (1 - d + 0.01), s / nb, i / nb, dl / nb,
                 100 * np.mean(np.abs(lens - 128) > 4), 100 * np.mean(lens > 132)))

    print("\n8. Marker overhead and payload information rate")
    for arm in G.ARMS:
        c = idx[arm][next(iter(pair_ids))]
        print("   %-16s payload %3d / 128 bases -> info rate %.2f, marker overhead %.2f"
              % (arm, c["payload_length"], c["payload_length"] / 128, c["marker_overhead"]))

    print("\n9. Marker appearing by chance inside the payload")
    for arm in ["marker_free", "marker_periodic"]:
        hits = sum(G.MARKER in c["payload_strand"] for c in idx[arm].values())
        print("   %-16s %d / %d payloads contain ACGTCAGT" % (arm, hits, len(idx[arm])))

    ex = idx["marker_periodic"][sorted(pair_ids)[0]]
    print("\n10. Example (marker_periodic, %s, deletion %.0f%%)" % (ex["pair_id"], ex["deletion_rate"] * 100))
    print("   clean : " + ex["clean_strand"])
    print("   read 0: " + ex["noisy_reads"][0] + "  (len %d)" % len(ex["noisy_reads"][0]))

    print("\n" + ("ALL CHECKS PASSED" if not FAILS else "%d CHECK(S) FAILED:\n  - " % len(FAILS) + "\n  - ".join(FAILS)))
    sys.exit(1 if FAILS else 0)


if __name__ == "__main__":
    main()


### Cell 4: get DNAFormer repo, pinned to the exact commit we checked

In [ ]:
!rm -rf {REPO}
!git clone -q https://github.com/itaiorr/Deep-DNA-based-storage {REPO}
!git -C {REPO} checkout -q 167a68261719ea458e79c40fcc485ae768c1a64a
!git -C {REPO} rev-parse HEAD
!ls {REPO}/DataGenerator

### Cell 5: tiny smoke test (about 10 seconds). Should end with ALL CHECKS PASSED.

In [ ]:
!rm -rf {TINY}
!python {CODE}/stage1_generate.py --repo {REPO} --out {TINY} --tiny
!python {CODE}/stage1_validate.py --repo {REPO} --data {TINY} --align-sample 10 | tail -3

### Cell 6: full pilot dataset (about 3-5 minutes, 181 MB, saved to Drive)

In [ ]:
!python {CODE}/stage1_generate.py --repo {REPO} --out {DATA}

### Cell 7: validate the pilot dataset. Paste the whole output of this cell back to Claude.

In [ ]:
!python {CODE}/stage1_validate.py --repo {REPO} --data {DATA} 2>&1 | tee {PROJECT}/logs/stage1_validation.txt
!du -sh {DATA}